# 02 — Model Training

Full training pipeline for the XGBoost stock direction predictor:
- Data ingestion → Feature engineering → TimeSeriesSplit → GridSearchCV
- Feature importance chart
- Hyperparameter comparison table
- MLflow run logging

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'ml-backend'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import f1_score, accuracy_score
from xgboost import XGBClassifier
import mlflow

from pipeline.ingest import fetch_stock_data
from pipeline.features import build_feature_matrix
from pipeline.evaluate import evaluate_stock_model

sns.set_theme(style='darkgrid')
TICKER = 'AAPL'
PERIOD = '2y'

## 1. Ingest & Feature Engineering

In [ ]:
raw_df = fetch_stock_data(TICKER, period=PERIOD, use_cache=True)
df, feature_cols = build_feature_matrix(raw_df)

X = df[feature_cols].values
y = df['target'].values

print(f'Dataset: {len(df)} samples | Features: {len(feature_cols)}')
print(f'Target balance — Up: {y.sum()} ({y.mean()*100:.1f}%)  Down: {(~y.astype(bool)).sum()}')
print(f'Feature columns: {feature_cols}')

## 2. TimeSeriesSplit — No Look-Ahead Bias

In [ ]:
# 80/20 chronological split
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Train: {len(X_train)} samples  |  Test: {len(X_test)} samples')
print(f'Train dates: {df["Date"].iloc[0].date()} → {df["Date"].iloc[split_idx-1].date()}')
print(f'Test  dates: {df["Date"].iloc[split_idx].date()} → {df["Date"].iloc[-1].date()}')

# Visualise the split
tscv = TimeSeriesSplit(n_splits=5)
fig, ax = plt.subplots(figsize=(14, 4))
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train)):
    ax.scatter(tr_idx,  [fold]*len(tr_idx),  marker='_', color='steelblue', s=40, label='Train' if fold==0 else '')
    ax.scatter(val_idx, [fold]*len(val_idx), marker='_', color='tomato',    s=40, label='Val'   if fold==0 else '')
ax.set_title('TimeSeriesSplit — 5 Folds (no shuffling)')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Fold')
ax.legend()
plt.tight_layout()
plt.show()

## 3. GridSearchCV Hyperparameter Tuning

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)

param_grid = {
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8],
}

base_model = XGBClassifier(random_state=42, eval_metric='logloss')
grid_search = GridSearchCV(base_model, param_grid, cv=tscv, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV F1: {grid_search.best_score_:.4f}')

## 4. Hyperparameter Comparison Table

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)
cols = [c for c in cv_results.columns if c.startswith('param_')] + ['mean_test_score', 'std_test_score', 'rank_test_score']
print(cv_results[cols].sort_values('rank_test_score').rename(columns={'mean_test_score':'mean_f1','std_test_score':'std_f1'}).to_string(index=False))

## 5. Evaluate on Holdout Test Set

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
metrics = evaluate_stock_model(y_test, y_pred)

print('Holdout metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

## 6. Feature Importance

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(importances.index, importances.values, color=plt.cm.Blues(importances.values / importances.max()))
ax.set_xlabel('XGBoost Feature Importance (gain)')
ax.set_title(f'{TICKER} — Feature Importance')
for bar, val in zip(bars, importances.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 7. Log Training Run to MLflow

In [ ]:
from mlflow_setup import log_training_run, init_experiments
init_experiments()

log_training_run(
    experiment_name='stock-predictor',
    params={**grid_search.best_params_, 'ticker': TICKER, 'period': PERIOD, 'source': 'notebook_02'},
    metrics=metrics,
    artifact_path=None,
    tag='manual',
)
print('Run logged to MLflow. View with: mlflow ui --backend-store-uri ml-backend/mlruns')